# SigAlg's `ProbabilityMeasure` class

In [1]:
# If running in Google Colab, uncomment the line below and run this cell first.
# Also, for Mac+Chrome users, beware of a known bug with LaTeX redering in Colab: https://github.com/googlecolab/colabtools/issues/3192

# !pip install sigalg

The `ProbabilityMeasure` class in SigAlg is the fundamental class for representing probability measures on sample spaces. The API reference is [here](https://johnmyers-phd.com/sigalg/api/modules/core/#sigalg.core.ProbabilityMeasure).

## Mathematical definition

Let $(\Omega, \mathcal{F})$ be a measurable space consisting of a $\sigma$-algebra $\mathcal{F}$ on a set $\Omega$. A *probability measure* $P$ is a countably additive function $P: \mathcal{F} \to [0,1]$ such that $P(\Omega) = 1$. Here, *countable additivity* means that

$$
P \left( \bigcup_{k=1}^\infty A_k \right) = \sum_{k=1}^\infty P(A_k)
$$

for all collections $\{A_k\}_{k=1}^\infty$ of pairwise disjoint measurable sets. If $\Omega$ is finite (as it always is, in SigAlg), then $P$ needs only to be finitely additive in order to be countably additive.

If $\mathcal{F}$ is the power set of a finite set $\Omega$, then $P$ is completely determined by its values on the finitely many singleton sets $\{\omega\}$ for $\omega \in \Omega$. In this case, we define

$$
P(\omega) \stackrel{\text{def}}{=} P(\{\omega\})
$$

for each $\omega\in \Omega$. From this viewpoint, $P:\Omega \to [0,1]$ functions as a *probability mass function* on $\Omega$.

In SigAlg, an instance `P` of `ProbabilityMeasure` represents such a probability measure.

## Creating probability measures

### From dictionaries

There are two methods for creating a `ProbabilityMeasure` from a dictionary. The first is to use the `from_atoms` method, which takes a dictionary mapping atom identifiers to the probability of each atom. To understand what this means, recall that an *atom* of a $\sigma$-algebra $\mathcal{F}$ is a non-empty event $A \in \mathcal{F}$ which is minimal with respect to subset inclusion, in the sense that if $B\subset A$ and $B \in \mathcal{F}$, then either $B = \emptyset$ or $B = A$. On a finite sample space $\Omega$, such a $\sigma$-algebra $\mathcal{F}$ determines (and is determined by) its atoms. If $\{A_i\}_{i\in I}$ is the collection of atoms, indexed by some finite set $I$, then every event in $\mathcal{F}$ is a disjoint union of some subcollection of the atoms. The indices in $I$ are called *atom identifiers*.

Let's begin by defining a sample space $\Omega = \{0, 1, 2, 3, 4\}$ and a $\sigma$-algebra $\mathcal{F}$ on $\Omega$ with atoms $A_0 = \{0, 1\}$, $A_1 = \{2\}$, and $A_2 = \{3, 4\}$. Notice that the set of atom identifiers is $I =\{0,1,2\}$.

In [2]:
from sigalg.core import SampleSpace, SigmaAlgebra

Omega = SampleSpace().from_sequence(size=5)
F = SigmaAlgebra(sample_space=Omega).from_dict(
    {
        0: 0,
        1: 0,
        2: 1,
        3: 2,
        4: 2,
    }
)

print(F)

Sigma algebra 'F':
        atom ID
sample         
0             0
1             0
2             1
3             2
4             2


As we mentioned, every event in $\mathcal{F}$ is a union of some subcollection of the atoms. By additivity, to define a probability measure $P$ with domain $\mathcal{F}$ it is then enough to specify the probabilities of the atoms. For example, we could specify that $P(A_0) = 0.5$, $P(A_1) = 0.3$, and $P(A_2) = 0.2$. This is exactly what the `from_atoms` method does: it takes a dictionary mapping atom identifiers to their probabilities, and returns a `ProbabilityMeasure` instance representing the corresponding probability measure.

In [3]:
from sigalg.core import ProbabilityMeasure

P = ProbabilityMeasure(sig_alg=F).from_atoms(
    {
        0: 0.5,
        1: 0.3,
        2: 0.2,
    }
)

print(P)

Probability measure 'P':
         probability
atom ID             
0                0.5
1                0.3
2                0.2


The second method for creating a `ProbabilityMeasure` from a dictionary is through the `from_dict` method, which takes a dictionary mapping sample points to their probabilities. This does not *mean* that the singletons of the sample points are necessarily measurable, which would be the case only if the $\sigma$-algebra is the power set of the sample space. For example, we can re-create the probability measure above as follows:

In [4]:
Q = ProbabilityMeasure(sig_alg=F, name="Q").from_dict(
    {
        0: 0.3,
        1: 0.2,
        2: 0.3,
        3: 0.05,
        4: 0.15,
    }
)

print(Q)

Probability measure 'Q':
         probability
atom ID             
0                0.5
1                0.3
2                0.2


Notice that the printout shows the same atom probabilities as the measure $P$ from above. So, $Q$ is actually the same probability measure as $P$, even though we created it from a different "pointwise" dictionary, instead of an "atomwise" dictionary. We can check equality of the measures as follows:

In [5]:
print(f"Are the measures P and Q equal? {P == Q}")

Are the measures P and Q equal? True


### From `pd.Series` objects

Rather than pass a dictionary to the `from_atoms` method, we can pass a `pd.Series` object to the `from_pandas` method with the atom identifiers as the index and the atom probabilities as the values. For example, we could re-create the probability measure $P$ from above as follows:

In [6]:
import pandas as pd

atom_probs = pd.Series([0.5, 0.3, 0.2])
R = ProbabilityMeasure(sig_alg=F, name="R").from_pandas(atom_probs)

print(R)

Probability measure 'R':
         probability
atom ID             
0                0.5
1                0.3
2                0.2


### The uniform distribution

The uniform distribution $U$ on a $\sigma$-algebra $\mathcal{F}$ is given by

$$
U(A) = \frac{|A|}{|\Omega|}
$$

for all events $A\in \mathcal{F}$, where $|A|$ is the cardinality of $A$ and $|\Omega|$ is the cardinality of the sample space. In SigAlg, uniform distributions are created through the `uniform` class method:

In [7]:
U = ProbabilityMeasure.uniform(  # use the same sigma-algebra F as above
    sig_alg=F, name="U"
)

print(U)

Probability measure 'U':
         probability
atom ID             
0                0.4
1                0.2
2                0.4


To understand the printout, recall that the atoms $A_0 = \{0, 1\}$ and $A_2 = \{3,4\}$ have cardinality $2$, while the atom $A_1 = \{2\}$ has cardinality $1$. Since the sample space $\Omega$ has cardinality $5$, the uniform distribution assigns probability $\frac{2}{5}$ to atoms $A_0$ and $A_2$, and probability $\frac{1}{5}$ to atom $A_1$.

### Random probability measures

It is often convenient to create a random probability measure. This can be done in SigAlg using the `from_rand` method, which takes an optional `random_state` argument for reproducibility:

In [8]:
import numpy as np

rng = np.random.default_rng(42)
P_random = ProbabilityMeasure(sig_alg=F, name="P_random").from_rand(random_state=rng)

print(P_random)

Probability measure 'P_random':
         probability
atom ID             
0           0.632779
1           0.318334
2           0.048887


## Calling probability measures

Mathematically, a probability measure $P$ is a function $P: \mathcal{F} \to [0,1]$. This is reflected in SigAlg by the fact that `ProbabilityMeasure` instances are callable.

For some examples, recall the $\sigma$-algebra $\mathcal{F}$ and probability measure $P$ we created above:

In [9]:
print(F, "\n")
print(P)

Sigma algebra 'F':
        atom ID
sample         
0             0
1             0
2             1
3             2
4             2 

Probability measure 'P':
         probability
atom ID             
0                0.5
1                0.3
2                0.2


Instances of `ProbabilityMeasure` may be called on instances of `Event`, which is SigAlg's class for representing events. For example, let's grab the atom $A_0=\{0,1\}$ from the $\sigma$-algebra and pass it to $P$ to compute its probability:

In [10]:
A0 = F.get_event([0, 1], name="A0")

print(A0, "\n")
print(f"P(A_0) = {P(A0)}")

Event 'A0':
[0, 1] 

P(A_0) = 0.5


The set $B = \{0, 1, 2\}$ is the union of the two atoms $A_0$ and $A_1$, and so is an event in $\mathcal{F}$. We could instantiate this event as we did above, by using the `get_event` method on `F`, but we can also pass in a list of sample points directly to `P`:

In [11]:
B = [0, 1, 2]

print(f"B = {B}\n")
print(f"P(B) = {P(B)}")

B = [0, 1, 2]

P(B) = 0.8


Notice that the atom $A_1 = \{2\}$ is a singleton. In SigAlg, if a singleton is measurable, then we can pass the sample point directly to the probability measure to compute its probability. For example:

In [12]:
print(f"P(2) = {P(2)}")

P(2) = 0.3


In the case that the $\sigma$-algebra is the power set, so that *all* singletons are measurable, we can pass any sample point directly to the probability measure to compute its probability and `P` is then essentially a probability mass function.

Passing non-measurable sets into a probability measure will raise an exception:

In [13]:
# Uncomment the line below and run this cell to see the exception raised when passing a non-measurable set into the probability measure.

# print(f"P([0, 2]) = {P([0, 2])}")

## Properties and attributes of probability measures

### $\sigma$-algebras and sample spaces

Instances of `ProbabilityMeasure` carry both the $\sigma$-algebra and the sample space on which they are defined as attributes `sig_alg` and `sample_space`, respectively:

In [14]:
print(P.sample_space, "\n")
print(P.sig_alg)

Sample space 'Omega':
[0, 1, 2, 3, 4] 

Sigma algebra 'F':
        atom ID
sample         
0             0
1             0
2             1
3             2
4             2


### Dictionaries and data

No matter how they were created, all `ProbabilityMeasure` instances also carry an `atom_probs` dictionary, mapping atom identifiers to the atom probabilities.

In [15]:
print(P.atom_probs)  # created using from_atoms
print(Q.atom_probs)  # created using from_dict
print(R.atom_probs)  # created using from_pandas
print(U.atom_probs)  # created using uniform
print(P_random.atom_probs)  # created using from_rand

{0: 0.5, 1: 0.3, 2: 0.2}
{0: 0.5, 1: 0.3, 2: 0.2}
{0: 0.5, 1: 0.3, 2: 0.2}
{0: 0.4, 1: 0.2, 2: 0.4}
{0: 0.6327794520305196, 1: 0.3183335399314935, 2: 0.048887008037987}


Likewise, all instances also carry a `data` attribute, which is a `pd.Series` object mapping atom identifiers to atom probabilities.

In [16]:
print(P.data, "\n")  # created using from_atoms
print(Q.data, "\n")  # created using from_dict
print(R.data, "\n")  # created using from_pandas
print(U.data, "\n")  # created using uniform
print(P_random.data)  # created using from_rand

atom ID
0    0.5
1    0.3
2    0.2
Name: probability, dtype: float64 

atom ID
0    0.5
1    0.3
2    0.2
Name: probability, dtype: float64 

atom ID
0    0.5
1    0.3
2    0.2
Name: probability, dtype: float64 

atom ID
0    0.4
1    0.2
2    0.4
Name: probability, dtype: float64 

atom ID
0    0.632779
1    0.318334
2    0.048887
Name: probability, dtype: float64


For instances that are created from "pointwise" dictionaries (which includes uniform and random measures), a dictionary mapping sample points to their probabilities is also stored as the `point_probs` attribute. For instances created directly from "atomwise" dictionaries, the `point_probs` attribute is set to `None`, since the probabilities of individual sample points are not well-defined in this case.

In [17]:
print(P.point_probs)  # created using from_atoms
print(Q.point_probs)  # created using from_dict
print(R.point_probs)  # created using from_pandas
print(U.point_probs)  # created using uniform
print(P_random.point_probs)  # created using from_rand

None
{0: 0.3, 1: 0.2, 2: 0.3, 3: 0.05, 4: 0.15}
None
{0: 0.2, 1: 0.2, 2: 0.2, 3: 0.2, 4: 0.2}
{0: np.float64(0.3209295336826644), 1: np.float64(0.31184991834785514), 2: np.float64(0.3183335399314935), 3: np.float64(0.03734877698258621), 4: np.float64(0.01153823105540079)}


## Probability methods

### Conditional probability

Given two events $A$ and $B$ with $P(B) > 0$, the *conditional probability* of $A$ given $B$ is

$$
P(A\mid B) = \frac{P(A \cap B)}{P(B)}.
$$

In SigAlg, conditional probabilities are computed by calling the `conditional_probability` method on an instance of `ProbabilityMeasure`, passing the two events as arguments:

In [18]:
Omega = SampleSpace().from_sequence(size=5)
F = SigmaAlgebra.power_set(Omega)

P = ProbabilityMeasure(sig_alg=F, name="P").from_dict(
    {
        0: 0.1,
        1: 0.2,
        2: 0.3,
        3: 0.25,
        4: 0.15,
    }
)

A = F.get_event([0, 1, 2], name="A")
B = F.get_event([1, 2, 3], name="B")

cond_prob = P.conditional_probability(event=A, given=B)
print(f"P(A|B) = {cond_prob}\n")

# Verify the formula
print(f"P(A ∩ B) = {P(A & B)}")
print(f"P(B) = {P(B)}")
print(f"P(A ∩ B) / P(B) = {P(A & B) / P(B)}")

P(A|B) = 0.6666666666666666

P(A ∩ B) = 0.5
P(B) = 0.75
P(A ∩ B) / P(B) = 0.6666666666666666


### Independence

Two events $A$ and $B$ are *independent* if $P(A \cap B) = P(A) P(B)$. In SigAlg, independence is checked by calling the `are_independent` method on an instance of `ProbabilityMeasure`, passing the two events as arguments:

In [19]:
from scipy.stats import bernoulli

from sigalg.core import Time
from sigalg.processes import IIDProcess

# Flip a biased coin twice
time = Time.discrete(length=1)
coin_flips = IIDProcess(
    distribution=bernoulli(p=0.6),
    support=[0, 1],
    name="coin_flips",
    time=time,
).from_enumeration()

Omega = coin_flips.domain
F = SigmaAlgebra.power_set(Omega)
P = coin_flips.prob_measure

print("Coin flips process:")
print(coin_flips, "\n")
print(P, "\n")

# Define events
first_heads = F.get_event([2, 3], name="first_heads")
second_heads = F.get_event([1, 3], name="second_heads")

print(
    f"Are 'first_heads' and 'second_heads' independent? {P.are_independent(event1=first_heads, event2=second_heads)}"
)

Coin flips process:
Stochastic process 'coin_flips':
time        0  1
trajectory      
0           0  0
1           0  1
2           1  0
3           1  1 

Probability measure 'P':
        probability
sample             
0              0.16
1              0.24
2              0.24
3              0.36 

Are 'first_heads' and 'second_heads' independent? True


Two random vectors $X$ and $Y$ are *independent* if the $\sigma$-algebras they generate are independent. The `are_independent` method can also be used to check independence of random vectors, by passing the two random vectors as arguments.

In [20]:
# Define random variables for the outcomes of the coin flips
flip1, flip2 = coin_flips

print(f"Are flip1 and flip2 independent? {P.are_independent(rv1=flip1, rv2=flip2)}\n")

# Create a dependent random variable
sum_flips = flip1 + flip2
print(f"Are flip1 and sum_flips independent? {P.are_independent(rv1=flip1, rv2=sum_flips)}")

TypeError: 'list' object is not callable

### Almost sure equality

Two random vectors $X$ and $Y$ on $\Omega$ are *equal almost surely* if

$$
P \left( \{\omega \in \Omega : X(\omega) \neq Y(\omega)\} \right) = 0.
$$

In [ ]:
from sigalg.core import RandomVariable

Omega = SampleSpace().from_sequence(size=4)
F = SigmaAlgebra.power_set(Omega)

P = ProbabilityMeasure(sig_alg=F, name="P").from_dict(
    {
        0: 0.4,
        1: 0.3,
        2: 0.3,
        3: 0.0,  # Zero probability
    }
)

X = RandomVariable(domain=Omega, name="X").from_dict(
    {
        0: 1,
        1: 2,
        2: 3,
        3: 4,
    }
)

Y = RandomVariable(domain=Omega, name="Y").from_dict(
    {
        0: 1,
        1: 2,
        2: 3,
        3: 100,  # Different from X, but on zero-probability event
    }
)

Z = RandomVariable(domain=Omega, name="Z").from_dict(
    {
        0: 1,
        1: 2,
        2: 10,  # Different from X on positive-probability event
        3: 4,
    }
)

print(f"P(X = Y almost surely)? {P.almost_surely_equal(X, Y)}")
print(f"P(X = Z almost surely)? {P.almost_surely_equal(X, Z)}")

P(X = Y almost surely)? True
P(X = Z almost surely)? False
